In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GMMHMM
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [5]:
# --------------------------
# Load and label data
# --------------------------
def load_stock_data(ticker='AAPL', start='2020-01-01', end='2023-01-01'):
    df = yf.download(ticker, start=start, end=end)
    df['frac_change'] = (df['Close'] - df['Open']) / df['Open']
    df['frac_high'] = (df['High'] - df['Open']) / df['Open']
    df['frac_low'] = (df['Open'] - df['Low']) / df['Open']
    df['label'] = (df['Close'] - df['Open']) / df['Open']
    δ = 0.001
    df['label'] = df['label'].apply(lambda x: 1 if x > δ else (-1 if x < -δ else 0))
    return df[['frac_change', 'frac_high', 'frac_low', 'label']].dropna()

In [10]:
# --------------------------
# Train HMM
# --------------------------
def train_hmm(X, n_states=4, n_mix=2):
    model = GMMHMM(n_components=n_states, n_mix=n_mix, covariance_type='diag', n_iter=100, min_covar=1e-4)
    model.fit(X)
    return model

# --------------------------
# Get soft state probabilities for each day
# --------------------------
def get_state_probabilities(model, X):
    logprob, posteriors = model.score_samples(X)
    return posteriors

In [6]:
# 1. Load & preprocess data
df = load_stock_data('AAPL')
features = df[['frac_change', 'frac_high', 'frac_low']].values
labels = df['label'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [11]:
# 2. Train GMM-HMM
hmm_model = train_hmm(X_scaled, n_states=4, n_mix=2)
# 3. Get posterior state probabilities (soft features)
posteriors = get_state_probabilities(hmm_model, X_scaled)